In [3]:
import cv2
import mediapipe as mp
import time
import numpy as np
import subprocess

BaseOptions = mp.tasks.BaseOptions
HandLandMarker = mp.tasks.vision.HandLandmarker
HandLandMarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

model_path = "/home/manish1209-op/DataScience/OpenCV/Models/hand_landmarker.task"

model = HandLandMarkerOptions(
    base_options = BaseOptions(model_asset_path = model_path),
    running_mode = RunningMode.VIDEO,
    num_hands = 1
)

detector = HandLandMarker.create_from_options(model)

start_time = time.monotonic()
prev_time = 0

cap = cv2.VideoCapture(0)

while True:
    ok, frame = cap.read()
    
    if not ok:
        print("Unable to read frame!")
        break
    
    frame = cv2.resize(frame, (960, 720))
    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    timestamp_ms = int((time.monotonic() - start_time) * 1000)
    
    current_time = time.monotonic()
    fps = 1 / (current_time - prev_time) if prev_time != 0 else 0
    prev_time = current_time
    
    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 0, 0),
        2
    )
    
    
    mp_images = mp.Image(
            image_format = mp.ImageFormat.SRGB,
            data = rgb
    )
    
    results = detector.detect_for_video(
        mp_images,
        timestamp_ms
    )
    
    
    Connection_points = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (5, 9), (9, 10), (10, 11), (11, 12),
        (9, 13), (13, 14), (14, 15), (15, 16),
        (0, 17), (13, 17), (17, 18), (18, 19),(19, 20) 
    ]
      
    height, width = frame.shape[:2]
    
   
        
    
    for hands in results.hand_landmarks:
        landmark_points = []
        for landmarks in hands:
            X = int(landmarks.x * width)
            Y = int(landmarks.y * height)
            
            landmark_points.append((X, Y))
            
            cv2.circle(
                frame,
                (X, Y), 
                5,
                (0, 0, 255),
                cv2.FILLED
            )
            
            
        
        for start, end in Connection_points:
            cv2.line(frame, landmark_points[start], landmark_points[end], (255, 255, 255), 2)
            
    if results.hand_landmarks:
        hand = results.hand_landmarks[0]
        thumb = hand[4]
        tx = int(thumb.x * width)            
        ty = int(thumb.y * height)
            
        index_fingertip = hand[8]
        ix = int(index_fingertip.x * width)
        iy = int(index_fingertip.y * height)
        
        cx = (ix + tx) // 2
        cy = (iy + ty) // 2
            
        cv2.circle(
            frame,
            (tx, ty),
            15,
            (0, 255, 0),
            cv2.FILLED
        )
            
        cv2.circle(
            frame,
            (ix, iy),
            13, 
            (0, 255, 0),
            cv2.FILLED
        )
        
        cv2.circle(
            frame,
            (cx, cy),
            13,
            (0, 255, 0),
            cv2.FILLED
        )
        
        cv2.line(frame, (ix, iy), (cx, cy), (0, 255, 0), 2)
        cv2.line(frame, (tx, ty), (cx, cy), (0, 255, 0), 2)
        
        dx = ix-tx
        dy = iy-ty
        distance = np.hypot(dx, dy)
    
        min_distance = 20
        max_distance = 200
    
        volume = ((distance-min_distance) /(max_distance-min_distance)) * 100
        volume = max(0, min(100, volume))
        volume = round(volume)
    
        subprocess.run(
            ["pactl", 
            "set-sink-volume",
            "@DEFAULT_SINK@",
            f"{volume}%"
        ])   
    
        cv2.putText(
            frame,
            f"Volume: {volume}%",
            (20, 100),
            cv2.FONT_HERSHEY_COMPLEX,
            1,
            (255, 0, 0),
            2
        )

        
    cv2.imshow("cam", frame)
    if cv2.waitKey(1) & 0xff == ord("q"):
        break
        
cap.release()
detector.close()
cv2.destroyAllWindows()


W0000 00:00:1793287411.937624    6890 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1793287411.965644    6890 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
QFontDatabase: Cannot find font directory /home/manish1209-op/DataScience/OpenCV/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/manish1209-op/DataScience/OpenCV/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/manish1209-op/DataScience/OpenCV/.venv/lib/python3.12/site-packages/cv2/qt/fonts